# Single-Cell RNA-seq Analysis with cachepy

**cachepy** is a disk-backed memoization decorator for Python.
This notebook demonstrates how cachepy accelerates a typical scRNA-seq pipeline
using the **PBMC 3k** dataset from 10x Genomics -- the same dataset used in the
[scanpy tutorial](https://scanpy-tutorials.readthedocs.io/en/latest/pbmc3k.html).

Each analysis step is wrapped with `@cache_file`, so re-running the notebook
skips expensive computations.  This is especially useful when:
- Iterating on downstream parameters (e.g. clustering resolution)
- Sharing pipelines across sessions
- Resuming after a crash

### Contents

| # | Section | What it shows |
|---|---------|---------------|
| 1 | [Load Data](#1.-Load-Data) | Download PBMC 3k dataset |
| 2 | [Quality Control](#2.-Quality-Control) | Cell / gene filtering |
| 3 | [Normalization & HVG](#3.-Normalization-and-HVG-Selection) | Log-normalize, select HVGs |
| 4 | [Dimensionality Reduction](#4.-Dimensionality-Reduction-(PCA-+-UMAP)) | PCA + UMAP |
| 5 | [Clustering](#5.-Clustering) | Leiden at multiple resolutions |
| 6 | [Marker Gene Detection](#6.-Marker-Gene-Detection) | Wilcoxon rank-sum |
| 7 | [Cell Type Annotation](#7.-Cell-Type-Annotation) | Rule-based PBMC labeling |
| 8 | [Visualization](#8.-Visualization) | UMAP plots |
| 9 | [Inspect the Cache](#9.-Inspect-the-Cache) | Stats & dependency graph |
| 10 | [Parameter Iteration](#10.-Parameter-Iteration) | Re-run with changed params |
| 11 | [Force Re-execution](#11.-Force-Re-execution) | `_force=True` bypass |
| 12 | [Speed Benchmark](#12.-Speed-Benchmark) | First execution vs cache hit |

In [ ]:
# --- Setup & Styling ---
import sys, time, shutil, warnings
from pathlib import Path

import numpy as np
import scanpy as sc

from IPython.display import HTML, display

display(HTML("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;500&display=swap');

body, .rendered_html, .text_cell_render {
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, 'Segoe UI', sans-serif !important;
    font-size: 15px !important;
    line-height: 1.65 !important;
    color: #1e293b !important;
}
.rendered_html h1 {
    font-weight: 700 !important; font-size: 2em !important;
    border-bottom: 3px solid #2563eb !important; padding-bottom: 0.3em !important;
    color: #0f172a !important;
}
.rendered_html h2 {
    font-weight: 600 !important; font-size: 1.4em !important;
    color: #1e40af !important; margin-top: 2em !important;
    border-bottom: 1px solid #e2e8f0 !important; padding-bottom: 0.2em !important;
}
.rendered_html h3 { font-weight: 600 !important; font-size: 1.1em !important; color: #334155 !important; }
.rendered_html code {
    font-family: 'JetBrains Mono', 'Fira Code', monospace !important;
    background: #f1f5f9 !important; padding: 2px 6px !important;
    border-radius: 4px !important; font-size: 0.88em !important; color: #7c3aed !important;
}
.rendered_html pre { font-family: 'JetBrains Mono', monospace !important; font-size: 0.88em !important; }
.rendered_html blockquote {
    border-left: 4px solid #2563eb !important; background: #eff6ff !important;
    padding: 12px 16px !important; margin: 12px 0 !important;
    border-radius: 0 8px 8px 0 !important; color: #1e40af !important;
}
.rendered_html table { border-collapse: collapse !important; margin: 12px 0 !important; font-size: 14px !important; }
.rendered_html th { background: #f1f5f9 !important; font-weight: 600 !important; }
.rendered_html th, .rendered_html td { padding: 6px 14px !important; border: 1px solid #e2e8f0 !important; }
div.input_area { border: 1px solid #e2e8f0 !important; border-radius: 6px !important; }
.CodeMirror { font-family: 'JetBrains Mono', monospace !important; font-size: 13.5px !important; }
</style>
"""))

sys.path.insert(0, str(Path.cwd().parent))

from cachepy import cache_file, cache_tree_nodes, cache_tree_reset
from cachepy.cache_file import cache_stats, _file_state_cache

warnings.filterwarnings("ignore", category=FutureWarning)

# --- Matplotlib style ---
import matplotlib.pyplot as plt
import matplotlib as mpl

COLORS = {
    "cold": "#dc2626",   # red  -- first execution
    "hot":  "#16a34a",   # green -- cache hit
    "accent": "#2563eb", # blue  -- speedup / general
    "muted": "#64748b",  # slate -- secondary
    "purple": "#7c3aed",
}

mpl.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["DejaVu Sans", "Helvetica Neue", "Arial", "sans-serif"],
    "font.size": 11,
    "axes.titlesize": 13,
    "axes.titleweight": 600,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.2,
    "figure.facecolor": "white",
    "figure.dpi": 100,
})

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, facecolor="white")

# --- Cache directory ---
CACHE_DIR = Path("scrnaseq_cache")

def fresh_start():
    if CACHE_DIR.exists():
        shutil.rmtree(CACHE_DIR)
    cache_tree_reset()
    _file_state_cache.clear()

# Uncomment the next line to clear the cache and re-run everything from scratch:
# fresh_start()

import logging
logging.basicConfig(level=logging.INFO, format="[cachepy] %(message)s", force=True)

print(f"scanpy {sc.__version__}")
print(f"Cache directory: {CACHE_DIR.resolve()}")

---
## 1. Load Data

Download the PBMC 3k dataset (2,700 cells, ~33k genes).

> **What to observe:** The first run downloads from the internet; subsequent runs load from cache in milliseconds.

In [ ]:
@cache_file(CACHE_DIR, verbose=True)
def load_pbmc3k():
    adata = sc.datasets.pbmc3k()
    adata.var_names_make_unique()
    return adata

t0 = time.time()
adata_raw = load_pbmc3k()
print(f"Loaded in {time.time()-t0:.2f}s: {adata_raw.shape[0]} cells x {adata_raw.shape[1]} genes")

> **Takeaway:** Data loading is cached -- no repeated downloads.

---
## 2. Quality Control

Filter cells and genes based on QC metrics.  The QC thresholds are parameters --
changing them re-triggers only this step and downstream, not the data loading.

> **What to observe:** Changing `min_genes`, `min_cells`, or `max_mito_pct` causes a cache miss for this step only.

In [ ]:
@cache_file(CACHE_DIR, verbose=True)
def run_qc(adata, min_genes=200, min_cells=3, max_mito_pct=5):
    """Filter cells and genes, annotate mitochondrial content."""
    adata = adata.copy()

    # Annotate mitochondrial genes
    adata.var["mt"] = adata.var_names.str.startswith("MT-")
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
    )

    # Filter
    n_before = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=min_genes)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    adata = adata[adata.obs.pct_counts_mt < max_mito_pct, :].copy()

    print(f"  QC: {n_before} -> {adata.n_obs} cells, {adata.n_vars} genes")
    return adata

t0 = time.time()
adata_qc = run_qc(adata_raw, min_genes=200, min_cells=3, max_mito_pct=5)
print(f"QC done in {time.time()-t0:.2f}s: {adata_qc.shape}")

---
## 3. Normalization and HVG Selection

> **What to observe:** `regress_out` and `scale` are expensive -- caching saves significant time on re-runs.

In [ ]:
@cache_file(CACHE_DIR, verbose=True)
def normalize_and_hvg(adata, n_top_genes=2000, target_sum=1e4):
    """Normalize, log-transform, and select highly variable genes."""
    adata = adata.copy()

    sc.pp.normalize_total(adata, target_sum=target_sum)
    sc.pp.log1p(adata)

    # Store raw counts for later (DE, plotting)
    adata.raw = adata

    sc.pp.highly_variable_genes(adata, n_top_genes=n_top_genes, flavor="seurat")
    print(f"  HVGs: {adata.var.highly_variable.sum()} genes selected")

    # Subset to HVGs
    adata = adata[:, adata.var.highly_variable].copy()

    # Regress out and scale
    sc.pp.regress_out(adata, ["total_counts", "pct_counts_mt"])
    sc.pp.scale(adata, max_value=10)

    return adata

t0 = time.time()
adata_norm = normalize_and_hvg(adata_qc, n_top_genes=2000)
print(f"Normalized in {time.time()-t0:.2f}s: {adata_norm.shape}")

---
## 4. Dimensionality Reduction (PCA + UMAP)

This is often the most expensive step. cachepy ensures it's only computed once.

> **What to observe:** PCA + neighbor graph + UMAP computed once, then instant on re-run.

In [ ]:
@cache_file(CACHE_DIR, verbose=True)
def run_dimred(adata, n_pcs=40, n_neighbors=10):
    """PCA, neighbor graph, and UMAP."""
    adata = adata.copy()

    sc.tl.pca(adata, n_comps=n_pcs, svd_solver="arpack")
    sc.pp.neighbors(adata, n_neighbors=n_neighbors, n_pcs=n_pcs)
    sc.tl.umap(adata)

    return adata

t0 = time.time()
adata_dimred = run_dimred(adata_norm, n_pcs=40, n_neighbors=10)
print(f"Dim reduction in {time.time()-t0:.2f}s")
print(f"PCA shape: {adata_dimred.obsm['X_pca'].shape}")
print(f"UMAP shape: {adata_dimred.obsm['X_umap'].shape}")

---
## 5. Clustering

This is where caching really shines -- you can iterate on the resolution
parameter without waiting for PCA/UMAP to recompute.

> **What to observe:** Only the first call for each resolution triggers execution; repeats are instant.

In [ ]:
@cache_file(CACHE_DIR, verbose=True)
def run_clustering(adata, resolution=1.0):
    """Leiden clustering at a given resolution."""
    adata = adata.copy()
    sc.tl.leiden(adata, resolution=resolution, flavor="igraph", n_iterations=2)
    n_clusters = adata.obs["leiden"].nunique()
    print(f"  Leiden (res={resolution}): {n_clusters} clusters")
    return adata

# Try different resolutions -- only the new ones compute
for res in [0.5, 1.0, 1.5]:
    t0 = time.time()
    adata_clust = run_clustering(adata_dimred, resolution=res)
    print(f"  ({time.time()-t0:.2f}s)\n")

> **Takeaway:** Upstream steps (load, QC, normalize, PCA) are never re-run -- only clustering with new resolutions executes.

---
## 6. Marker Gene Detection

> **What to observe:** Marker detection is cached per clustering result. Changing resolution upstream produces a new input, triggering a fresh marker search.

In [ ]:
# Use resolution=1.0 for downstream
adata_final = run_clustering(adata_dimred, resolution=1.0)

@cache_file(CACHE_DIR, verbose=True)
def find_markers(adata, n_genes=25, method="wilcoxon"):
    """Find marker genes per cluster."""
    adata = adata.copy()
    sc.tl.rank_genes_groups(adata, groupby="leiden", method=method, n_genes=n_genes)
    return adata

t0 = time.time()
adata_markers = find_markers(adata_final, n_genes=25)
print(f"Markers found in {time.time()-t0:.2f}s")

# Show top 3 markers per cluster (first 5 clusters)
result = adata_markers.uns["rank_genes_groups"]
groups = result["names"].dtype.names
print(f"\nTop 3 markers per cluster (first 5 clusters):")
for g in groups[:5]:
    genes = [result["names"][g][i] for i in range(3)]
    print(f"  Cluster {g}: {', '.join(genes)}")

---
## 7. Cell Type Annotation

Assign cell types based on known PBMC markers.

> **What to observe:** Annotation uses cached marker results -- only the labeling logic runs.

In [ ]:
@cache_file(CACHE_DIR, verbose=True)
def annotate_celltypes(adata):
    """Simple rule-based annotation using canonical PBMC markers."""
    adata = adata.copy()

    # Canonical PBMC marker mapping
    marker_map = {
        "CD4 T":    ["IL7R", "CD4"],
        "CD8 T":    ["CD8A", "CD8B"],
        "B":        ["MS4A1", "CD79A"],
        "NK":       ["GNLY", "NKG7"],
        "Mono":     ["CD14", "LYZ"],
        "DC":       ["FCER1A", "CST3"],
        "Platelet": ["PPBP"],
    }

    # Score each cluster by mean expression of markers
    raw_df = adata.raw.to_adata()
    cluster_labels = {}

    for cluster in adata.obs["leiden"].unique():
        mask = adata.obs["leiden"] == cluster
        best_score = -1
        best_type = "Unknown"

        for ctype, markers in marker_map.items():
            present = [m for m in markers if m in raw_df.var_names]
            if not present:
                continue
            score = np.mean(raw_df[mask][:, present].X.toarray())
            if score > best_score:
                best_score = score
                best_type = ctype

        cluster_labels[cluster] = best_type

    adata.obs["cell_type"] = adata.obs["leiden"].map(cluster_labels).astype("category")
    return adata

t0 = time.time()
adata_annotated = annotate_celltypes(adata_markers)
print(f"Annotated in {time.time()-t0:.2f}s")
print("\nCell type counts:")
print(adata_annotated.obs["cell_type"].value_counts().to_string())

---
## 8. Visualization

> **What to observe:** All data is already in memory (from cache). Plotting is instant.

In [ ]:
sc.pl.umap(adata_annotated, color=["leiden", "cell_type"], wspace=0.4)

In [ ]:
canonical_markers = ["IL7R", "CD8A", "MS4A1", "GNLY", "CD14", "FCER1A", "PPBP"]
sc.pl.umap(adata_annotated, color=canonical_markers, ncols=4, wspace=0.3)

In [ ]:
sc.pl.rank_genes_groups(adata_markers, n_genes=10, sharey=False)

---
## 9. Inspect the Cache

See what's been cached and the dependency graph.

> **What to observe:** Each pipeline step has one cache entry; the graph tracks parent/child relationships.

In [ ]:
# Cache statistics
stats = cache_stats(CACHE_DIR)
print("Cache statistics:")
print(f"  Entries:    {stats['n_entries']}")
print(f"  Total size: {stats['total_size_mb']:.1f} MB")

print(f"\nCached files:")
for f in sorted(CACHE_DIR.glob("*.pkl")):
    size_kb = f.stat().st_size / 1024
    fname = f.stem.split(".")[0]
    print(f"  {fname:30s}  {size_kb:8.1f} KB")

# Dependency graph
nodes = cache_tree_nodes()
print(f"\nDependency graph: {len(nodes)} nodes")
for nid, node in nodes.items():
    parents = len(node.get('parents', []))
    children = len(node.get('children', []))
    print(f"  {node['fname']:30s}  parents={parents}  children={children}")

> **Takeaway:** cachepy tracks not just *what* is cached but *how* functions depend on each other.

---
## 10. Parameter Iteration

The key benefit: change a downstream parameter and only that step re-runs.
Everything upstream is cached.

> **What to observe:** Steps 1-4 are cache hits (instant). Only step 5 with the new resolution actually executes.

In [ ]:
print("Re-running the full pipeline with a different clustering resolution...\n")

t0 = time.time()
# Steps 1-4 are all cache hits (instant)
raw = load_pbmc3k()
qc = run_qc(raw)
norm = normalize_and_hvg(qc)
dimred = run_dimred(norm)
t_cached = time.time() - t0
print(f"Steps 1-4 (from cache): {t_cached:.2f}s\n")

# Only step 5 re-runs with new resolution
t0 = time.time()
clust_new = run_clustering(dimred, resolution=0.3)
t_new = time.time() - t0
print(f"Step 5 (resolution=0.3): {t_new:.2f}s")
print(f"\nTotal: {t_cached + t_new:.2f}s instead of re-running everything")

In [ ]:
# Compare clustering resolutions side by side
sc.pl.umap(clust_new, color="leiden", title="resolution = 0.3")
sc.pl.umap(adata_annotated, color="leiden", title="resolution = 1.0")

> **Takeaway:** Iterating on clustering resolution costs only the clustering step itself -- everything upstream is free.

---
## 11. Force Re-execution

If upstream data changes (e.g. new reference genome), use `_force=True` to bypass the cache.

> **What to observe:** Despite being cached, QC re-runs because of `_force=True`.

In [ ]:
print("Forcing QC to re-run with stricter thresholds:\n")

t0 = time.time()
qc_strict = run_qc(raw, min_genes=300, min_cells=5, max_mito_pct=3)
print(f"\nStricter QC: {qc_strict.shape} in {time.time()-t0:.2f}s")

> **Takeaway:** `_force=True` lets you selectively re-execute any step without clearing the entire cache.

---
## 12. Speed Benchmark

Measure the wall-clock time for each pipeline step -- first execution (cold)
vs cache hit (warm) -- and plot the difference.

> **What to observe:** Cache overhead is constant (~1-10 ms) regardless of original computation time.

In [ ]:
step_names = [
    "Load data", "QC", "Normalize\n+ HVG",
    "PCA +\nUMAP", "Clustering", "Markers", "Annotate",
]

# --- First execution (cold) ---
fresh_start()
_file_state_cache.clear()

first_times = []
results = []

# Step 0: load
t0 = time.time()
res = load_pbmc3k()
first_times.append(time.time() - t0)
results.append(res)

# Step 1: QC
t0 = time.time()
res = run_qc(results[0], min_genes=200, min_cells=3, max_mito_pct=5)
first_times.append(time.time() - t0)
results.append(res)

# Step 2: normalize
t0 = time.time()
res = normalize_and_hvg(results[1], n_top_genes=2000)
first_times.append(time.time() - t0)
results.append(res)

# Step 3: dimred
t0 = time.time()
res = run_dimred(results[2], n_pcs=40, n_neighbors=10)
first_times.append(time.time() - t0)
results.append(res)

# Step 4: clustering
t0 = time.time()
res = run_clustering(results[3], resolution=1.0)
first_times.append(time.time() - t0)
results.append(res)

# Step 5: markers
t0 = time.time()
res = find_markers(results[4], n_genes=25)
first_times.append(time.time() - t0)
results.append(res)

# Step 6: annotate
t0 = time.time()
res = annotate_celltypes(results[5])
first_times.append(time.time() - t0)
results.append(res)

# --- Cache hits (warm) ---
_file_state_cache.clear()
cache_times = []

t0 = time.time(); load_pbmc3k();                                    cache_times.append(time.time() - t0)
t0 = time.time(); run_qc(results[0]);                               cache_times.append(time.time() - t0)
t0 = time.time(); normalize_and_hvg(results[1]);                    cache_times.append(time.time() - t0)
t0 = time.time(); run_dimred(results[2]);                            cache_times.append(time.time() - t0)
t0 = time.time(); run_clustering(results[3], resolution=1.0);       cache_times.append(time.time() - t0)
t0 = time.time(); find_markers(results[4], n_genes=25);             cache_times.append(time.time() - t0)
t0 = time.time(); annotate_celltypes(results[5]);                   cache_times.append(time.time() - t0)

# --- Plot ---
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

x = np.arange(len(step_names))
w = 0.35

ax = axes[0]
ax.bar(x - w/2, first_times,  w, label="First execution", color=COLORS["cold"])
ax.bar(x + w/2, cache_times, w, label="Cache hit",        color=COLORS["hot"])
ax.set_xlabel("Pipeline step")
ax.set_ylabel("Time (seconds)")
ax.set_title("scRNA-seq Pipeline: First Run vs Cache Hit")
ax.set_xticks(x)
ax.set_xticklabels(step_names, fontsize=9)
ax.legend(frameon=True, fancybox=True)
ax.set_yscale("log")

speedups = [f / max(c, 1e-6) for f, c in zip(first_times, cache_times)]
ax2 = axes[1]
ax2.bar(x, speedups, color=COLORS["accent"], width=0.5)
ax2.set_xlabel("Pipeline step")
ax2.set_ylabel("Speedup (x)")
ax2.set_title("Cache Speedup per Step")
ax2.set_xticks(x)
ax2.set_xticklabels(step_names, fontsize=9)
for i, s in enumerate(speedups):
    ax2.text(i, s + max(speedups) * 0.03, f"{s:.0f}x",
             ha="center", fontweight="bold", fontsize=11)

fig.tight_layout()
plt.show()

total_first  = sum(first_times)
total_cached = sum(cache_times)
print(f"\nTotal pipeline: {total_first:.1f}s (first) vs {total_cached:.2f}s (cached)"
      f" = {total_first/total_cached:.0f}x speedup")

> **Takeaway:** Cache overhead is negligible. The speedup grows linearly with computation cost.

---
## Cleanup

Uncomment to remove all cached data.

In [ ]:
# fresh_start()
# print("Cache cleared.")

logging.getLogger().setLevel(logging.WARNING)
print(f"Cache preserved at: {CACHE_DIR.resolve()}")